# 05 · Learning from preferences: reward models and DPO

**Track:** Training LLMs & Frontier Research · **Runtime (CPU default):** ~2 minutes · **GPU:** not needed

SFT teaches a model to imitate good answers. But for many prompts there is no single "right" answer to imitate, while
people can easily say **which of two answers is better**. RLHF (InstructGPT, Ouyang et al. 2022) turns those pairwise
judgments into a **reward model** and then optimises the policy against it with PPO. **DPO** (Rafailov et al. 2023)
showed that you can skip the explicit reward model and the RL loop: the policy itself defines an *implicit* reward,
and a simple classification-style loss on preference pairs optimises the same KL-regularised objective. In this
notebook we do both on a 36-pair preference dataset: train a Bradley–Terry reward model with TRL's `RewardTrainer`,
train a policy with `DPOTrainer`, derive the DPO loss by hand and check it numerically, and then let the reward model
judge the DPO policy's generations.

> **Source & license credit.** Uses Hugging Face **[TRL](https://github.com/huggingface/trl)** (`RewardTrainer`,
> `DPOTrainer`; Apache-2.0), **[PEFT](https://github.com/huggingface/peft)**, **[transformers](https://github.com/huggingface/transformers)**
> and **[datasets](https://github.com/huggingface/datasets)** (Apache-2.0). Model:
> **[HuggingFaceTB/SmolLM2-135M-Instruct](https://huggingface.co/HuggingFaceTB/SmolLM2-135M-Instruct)** (Apache-2.0),
> downloaded at runtime. The 36 preference pairs in [`research/data/preferences_tiny.jsonl`](../data/preferences_tiny.jsonl)
> were written for this repo (original). The manual loss functions are original implementations of the equations in the
> cited papers. See [`research/THIRD_PARTY_NOTICES.md`](../THIRD_PARTY_NOTICES.md).

## Learning goals

1. Write down the **Bradley–Terry** preference model and the reward-model loss used in InstructGPT.
2. Train a reward model (LM backbone + scalar head) and measure **pairwise accuracy** on held-out pairs.
3. Derive **DPO** from the KL-regularised RL objective, implement its loss by hand, and check it against numbers we
   compute ourselves.
4. Train a policy with `DPOTrainer` (LoRA; the reference model is simply "adapter off") and read its metrics:
   implicit rewards, margins, accuracies.
5. Evaluate honestly: implicit-reward accuracy on held-out pairs, and RM-scored generations before vs. after.

## Background

* Christiano et al., *Deep RL from Human Preferences*, 2017. [arXiv:1706.03741](https://arxiv.org/abs/1706.03741)
* Stiennon et al., *Learning to summarize from human feedback*, 2020. [arXiv:2009.01325](https://arxiv.org/abs/2009.01325)
* Ouyang et al., *Training language models to follow instructions with human feedback* (InstructGPT), 2022.
  [arXiv:2203.02155](https://arxiv.org/abs/2203.02155)
* Bai et al., *Training a Helpful and Harmless Assistant with RLHF*, Anthropic 2022. [arXiv:2204.05862](https://arxiv.org/abs/2204.05862)
* Rafailov, Sharma, Mitchell, Ermon, Manning, Finn, *Direct Preference Optimization: Your Language Model is Secretly a
  Reward Model*, NeurIPS 2023. [arXiv:2305.18290](https://arxiv.org/abs/2305.18290)
* Gao, Schulman, Hilton, *Scaling Laws for Reward Model Overoptimization*, 2022. [arXiv:2210.10760](https://arxiv.org/abs/2210.10760)

### The math in four lines

**Bradley–Terry.** $P(y_w \succ y_l \mid x) = \sigma\big(r(x,y_w) - r(x,y_l)\big)$. The reward-model loss is the negative
log-likelihood of the human choices: $\mathcal{L}_{RM} = -\mathbb{E}\,\log\sigma\big(r_\phi(x,y_w) - r_\phi(x,y_l)\big)$.

**RLHF objective.** $\max_\pi\; \mathbb{E}_{x,\,y\sim\pi}[r(x,y)] - \beta\,\mathrm{KL}\big(\pi(\cdot|x)\,\|\,\pi_{ref}(\cdot|x)\big)$.
Its optimum is $\pi^*(y|x) = \frac{1}{Z(x)}\pi_{ref}(y|x)\,e^{r(x,y)/\beta}$.

**Invert it.** $r(x,y) = \beta \log\frac{\pi^*(y|x)}{\pi_{ref}(y|x)} + \beta\log Z(x)$. The intractable $Z(x)$ cancels in a
*difference* of rewards, which is all Bradley–Terry needs.

**DPO loss.** $\mathcal{L}_{DPO} = -\mathbb{E}\,\log\sigma\Big(\beta\big[\log\frac{\pi_\theta(y_w|x)}{\pi_{ref}(y_w|x)} - \log\frac{\pi_\theta(y_l|x)}{\pi_{ref}(y_l|x)}\big]\Big)$.
The bracketed log-ratios times $\beta$ are the **implicit rewards** that TRL logs as `rewards/chosen` and `rewards/rejected`.

In [ ]:
# --- Setup: make research/src importable (from research/, research/notebooks/, or a Colab clone) ---
import sys, pathlib
for _p in [pathlib.Path.cwd().resolve(), *pathlib.Path.cwd().resolve().parents]:
    _src = next((c for c in (_p / "src", _p / "research" / "src") if (c / "lab_config.py").exists()), None)
    if _src:
        sys.path.insert(0, str(_src))
        break
else:
    raise RuntimeError("Could not find research/src -- run this notebook from inside the repo.")

from lab_config import CFG, pick, set_seed, heavy_cell, read_text, quiet_hf_logging

FULL_SIZE = CFG.full_size   # <- set True here (or export LAB_FULL_SIZE=1) for the full-size run
CFG.full_size = FULL_SIZE
set_seed()
print(CFG.summary())
quiet_hf_logging()

In [ ]:
import json, random, time
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSequenceClassification
from peft import LoraConfig
from trl import RewardTrainer, RewardConfig, DPOTrainer, DPOConfig

MODEL = pick("HuggingFaceTB/SmolLM2-135M-Instruct", "HuggingFaceTB/SmolLM2-360M-Instruct")
tok = AutoTokenizer.from_pretrained(MODEL)
tok.pad_token = "<|endoftext|>"     # a pad token distinct from <|im_end|> (the chat EOS)

pairs = [json.loads(l) for l in (CFG.data_dir / "preferences_tiny.jsonl").read_text().splitlines()]
random.Random(CFG.seed).shuffle(pairs)
n_test = 8
train_pairs, test_pairs = pairs[n_test:], pairs[:n_test]
print(f"{len(train_pairs)} train pairs, {len(test_pairs)} test pairs")
print("categories (test):", [p["category"] for p in test_pairs])
print(json.dumps(train_pairs[0], indent=1))

def to_conversational(p):
    return {"prompt": [{"role": "user", "content": p["prompt"]}],
            "chosen": [{"role": "assistant", "content": p["chosen"]}],
            "rejected": [{"role": "assistant", "content": p["rejected"]}]}

train_ds = Dataset.from_list([to_conversational(p) for p in train_pairs])

## Part A · A Bradley–Terry reward model

### A1. The loss, by hand

Before calling a trainer, here is the whole RM objective on made-up scores: two pairs, where the model is right about
the first (chosen scores higher) and wrong about the second.

In [ ]:
def bradley_terry_loss(r_chosen, r_rejected):
    return -F.logsigmoid(r_chosen - r_rejected).mean()

r_c = torch.tensor([2.0, -0.5]); r_r = torch.tensor([0.5, 1.0])
print("P(chosen > rejected):", torch.sigmoid(r_c - r_r).numpy().round(3))
print("BT loss:", round(bradley_terry_loss(r_c, r_r).item(), 4))
print("chance-level loss (equal scores) = ln 2 =", round(np.log(2), 4))

### A2. Train it with TRL

The reward model is the same transformer with its LM head replaced by a scalar "score" head, read out at the last
token. As in InstructGPT, it starts from the SFT'd model (here, SmolLM2-Instruct), which already "understands" the
responses. We fine-tune with LoRA plus the new head.

In [ ]:
rm = AutoModelForSequenceClassification.from_pretrained(MODEL, num_labels=1).to(CFG.device)
rm.config.pad_token_id = tok.pad_token_id

@torch.no_grad()
def rm_score(model, prompt, response):
    text = tok.apply_chat_template([{"role": "user", "content": prompt}, {"role": "assistant", "content": response}], tokenize=False)
    enc = tok(text, return_tensors="pt").to(CFG.device)
    return model(**enc).logits[0, 0].item()

def pairwise_accuracy(score_fn, pairs):
    margins = [score_fn(p["prompt"], p["chosen"]) - score_fn(p["prompt"], p["rejected"]) for p in pairs]
    return float(np.mean([m > 0 for m in margins])), margins

rm.eval()
acc0, _ = pairwise_accuracy(lambda x, y: rm_score(rm, x, y), test_pairs)
print(f"untrained RM (random score head) held-out pairwise accuracy: {acc0:.0%}")

rm_args = RewardConfig(
    output_dir=str(CFG.output_dir / "rm_run"),
    max_steps=pick(28, 150), per_device_train_batch_size=4,
    learning_rate=pick(5e-4, 1e-4), lr_scheduler_type="cosine", warmup_steps=2,
    max_length=256, logging_steps=4, save_strategy="no", report_to=[], disable_tqdm=True,
    seed=CFG.seed, use_cpu=not CFG.has_gpu,
)
rm_trainer = RewardTrainer(
    model=rm, args=rm_args, train_dataset=train_ds, processing_class=tok,
    peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, task_type="SEQ_CLS",
                           target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]),
)
t0 = time.time()
rm_trainer.train()
print(f"reward model trained in {time.time() - t0:.0f}s")

In [ ]:
reward_model = rm_trainer.model.eval()
score = lambda x, y: rm_score(reward_model, x, y)
acc_train, _ = pairwise_accuracy(score, train_pairs)
acc_test, test_margins = pairwise_accuracy(score, test_pairs)
print(f"pairwise accuracy: train {acc_train:.0%} | held-out {acc_test:.0%}  (chance = 50%, n_test = {len(test_pairs)})")
for p, m in zip(test_pairs, test_margins):
    print(f"  {'✓' if m > 0 else '✗'} margin {m:+6.2f}  [{p['category']:12}] {p['prompt'][:60]}")

logs = [h for h in rm_trainer.state.log_history if "loss" in h]
plt.figure(figsize=(5, 3))
plt.plot([h["step"] for h in logs], [h["loss"] for h in logs], "o-", label="BT loss")
plt.axhline(np.log(2), ls="--", c="gray", lw=1, label="ln 2 (chance)")
plt.xlabel("step"); plt.legend(); plt.title("Reward model training"); plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

**Caveat.** Eight held-out pairs means one pair = 12.5 percentage points: an accuracy difference of a pair or two is
noise. Compare with the *untrained* RM's accuracy printed above: a random score head can land well above 50% on 8
pairs by luck alone. Also, our rejected answers are often *obviously* bad (rude, wrong, or unsafe), which makes this much easier
than real preference data, where both answers are usually plausible and annotators agree only ~65–75% of the time
(InstructGPT reports ~73% inter-annotator agreement). With so little data, the RM may also latch onto **surface
features** like length or tone. Check the margins on the "concision" pairs, where the rejected answer is the longer one.

## Part B · Direct Preference Optimization

### B1. Sequence log-probabilities and the DPO loss, by hand

DPO needs $\log \pi(y\mid x) = \sum_t \log \pi(y_t \mid x, y_{<t})$ for the response tokens under the policy and under
the frozen reference. With LoRA, the reference is just the same network with the adapter **disabled**, so there is no
second copy of the weights.

In [ ]:
policy = AutoModelForCausalLM.from_pretrained(MODEL).to(CFG.device)
BETA = 0.1

def response_logprob(model, prompt, response):
    """Sum of log-probs of `response` tokens given the chat-formatted `prompt` (no grad)."""
    ctx = tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    ctx_ids = tok(ctx, return_tensors="pt", add_special_tokens=False)["input_ids"]
    full_ids = tok(ctx + response + "<|im_end|>", return_tensors="pt", add_special_tokens=False)["input_ids"].to(CFG.device)
    with torch.no_grad():
        logits = model(full_ids).logits[0, :-1].float()
    lp = torch.log_softmax(logits, -1).gather(-1, full_ids[0, 1:, None]).squeeze(-1)
    return lp[ctx_ids.shape[1] - 1:].sum().item()

def dpo_loss(pi_c, pi_r, ref_c, ref_r, beta=BETA):
    """Rafailov et al. (2023), eq. 7. Inputs are sequence log-probs (tensors)."""
    logits = beta * ((pi_c - ref_c) - (pi_r - ref_r))
    return -F.logsigmoid(logits).mean(), beta * (pi_c - ref_c), beta * (pi_r - ref_r)

# Sanity check at initialisation: policy == reference, so every implicit reward is 0 and the loss is ln 2.
p = train_pairs[0]
lp_c, lp_r = response_logprob(policy, p["prompt"], p["chosen"]), response_logprob(policy, p["prompt"], p["rejected"])
loss, rc, rr = dpo_loss(torch.tensor([lp_c]), torch.tensor([lp_r]), torch.tensor([lp_c]), torch.tensor([lp_r]))
print(f"log π(chosen)={lp_c:.1f}  log π(rejected)={lp_r:.1f}")
print(f"DPO loss at init = {loss.item():.4f} (ln 2 = {np.log(2):.4f}); implicit rewards {rc.item():.2f}, {rr.item():.2f}")

Note that the raw log-probabilities are *not* a reward: a short, rude answer can easily be more likely than a long,
helpful one. DPO only ever looks at how much the policy has moved **relative to the reference** on each response.

### B2. Train with `DPOTrainer`

In [ ]:
dpo_args = DPOConfig(
    output_dir=str(CFG.output_dir / "dpo_run"),
    beta=BETA,
    max_steps=pick(28, 200), per_device_train_batch_size=4,
    learning_rate=pick(5e-4, 5e-5), lr_scheduler_type="cosine", warmup_steps=2,
    max_length=256, logging_steps=4, save_strategy="no", report_to=[], disable_tqdm=True,
    seed=CFG.seed, use_cpu=not CFG.has_gpu,
)
dpo_trainer = DPOTrainer(
    model=policy, ref_model=None, args=dpo_args, train_dataset=train_ds, processing_class=tok,
    peft_config=LoraConfig(r=16, lora_alpha=32, lora_dropout=0.0, task_type="CAUSAL_LM",
                           target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]),
)
t0 = time.time()
dpo_trainer.train()
print(f"DPO finished in {time.time() - t0:.0f}s")

logs = [h for h in dpo_trainer.state.log_history if "rewards/margins" in h]
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
ax[0].plot([h["step"] for h in logs], [h["loss"] for h in logs], "o-", label="DPO loss")
ax[0].axhline(np.log(2), ls="--", c="gray", lw=1, label="ln 2"); ax[0].legend()
ax[1].plot([h["step"] for h in logs], [h["rewards/chosen"] for h in logs], "o-", label="implicit reward: chosen")
ax[1].plot([h["step"] for h in logs], [h["rewards/rejected"] for h in logs], "o-", label="implicit reward: rejected")
ax[1].legend()
for a in ax:
    a.set_xlabel("step"); a.grid(alpha=0.3)
plt.tight_layout(); plt.show()

A common pattern (also reported at scale) is that **both** implicit rewards go down, with the rejected one going down
faster. DPO increases the *margin*, not necessarily the absolute likelihood of the chosen response. That is one
motivation for variants such as IPO (Azar et al., 2023, [arXiv:2310.12036](https://arxiv.org/abs/2310.12036)) and
DPO-Positive (Pal et al., 2024, [arXiv:2402.13228](https://arxiv.org/abs/2402.13228)).

### B3. Check TRL against our own implementation on held-out pairs

In [ ]:
dpo_policy = dpo_trainer.model.eval()

def implicit_rewards(pairs):
    rows = []
    for p in pairs:
        pi_c, pi_r = response_logprob(dpo_policy, p["prompt"], p["chosen"]), response_logprob(dpo_policy, p["prompt"], p["rejected"])
        with dpo_policy.disable_adapter():
            ref_c, ref_r = response_logprob(dpo_policy, p["prompt"], p["chosen"]), response_logprob(dpo_policy, p["prompt"], p["rejected"])
        rows.append((pi_c, pi_r, ref_c, ref_r))
    return torch.tensor(rows)

R = implicit_rewards(test_pairs)
loss, rc, rr = dpo_loss(R[:, 0], R[:, 1], R[:, 2], R[:, 3])
print(f"held-out DPO loss {loss.item():.3f} (ln 2 = {np.log(2):.3f} at init)")
print(f"held-out implicit-reward accuracy: {(rc > rr).float().mean().item():.0%}   mean margin {(rc - rr).mean().item():+.3f}")
for p, a, b in zip(test_pairs, rc, rr):
    print(f"  {'✓' if a > b else '✗'} r_c={a.item():+.2f} r_r={b.item():+.2f} [{p['category']:12}] {p['prompt'][:55]}")

### B4. Does the DPO policy generate answers the reward model likes better?

Now the two halves meet: the reward model from Part A (trained on the same preferences, independently) scores
greedy generations from the reference policy (adapter off) and the DPO policy (adapter on), on the held-out prompts.
With 8 prompts and a 135M model this is a **smoke test**, not an evaluation. Any difference is suggestive at best.

In [ ]:
@torch.no_grad()
def generate(model, prompt, max_new_tokens=60):
    ctx = tok.apply_chat_template([{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True)
    enc = tok(ctx, return_tensors="pt", add_special_tokens=False).to(CFG.device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, repetition_penalty=1.1,
                         pad_token_id=tok.pad_token_id, eos_token_id=tok.convert_tokens_to_ids("<|im_end|>"))
    return tok.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True).strip()

results = []
for p in test_pairs:
    with dpo_policy.disable_adapter():
        ref_out = generate(dpo_policy, p["prompt"])
    dpo_out = generate(dpo_policy, p["prompt"])
    results.append((p["prompt"], ref_out, dpo_out, score(p["prompt"], ref_out), score(p["prompt"], dpo_out)))

ref_mean, dpo_mean = np.mean([r[3] for r in results]), np.mean([r[4] for r in results])
print(f"mean RM score: reference {ref_mean:+.2f} | DPO {dpo_mean:+.2f} | DPO wins on {sum(r[4] > r[3] for r in results)}/{len(results)} prompts\n")
for prompt, ref_out, dpo_out, s_ref, s_dpo in results:
    print("=" * 90)
    print("PROMPT:", prompt)
    print(f"[ref {s_ref:+.2f}] {ref_out[:220]}")
    print(f"[dpo {s_dpo:+.2f}] {dpo_out[:220]}")

**Interpretation guardrails.** First, *read the generations* rather than trusting the mean score: check whether the
RM ever gives a high score to an answer that is wrong or unsafe. (In our reference CPU run it did: on "How many
days are in a week?" the DPO policy's wrong answer scored well above the reference model's correct "7 days".) A learned reward is a proxy,
and a 28-pair proxy is a very leaky one.
* The RM and DPO were trained on the *same* 28 pairs, so they share the same biases; agreement between them is not
  independent evidence of quality. Real pipelines evaluate with held-out human ratings or a stronger judge.
* Optimising a policy hard against a learned RM eventually **over-optimises** it: the proxy reward keeps going up
  while true quality goes down (Gao et al., 2022). The KL term (β) is what keeps the policy near the reference.

## Your turn

1. **β sweep.** Train DPO with `beta ∈ {0.01, 0.1, 0.5}`. How do the margins and the KL to the reference change?
   (Estimate KL on generations as the mean of $\log\pi_\theta(y|x) - \log\pi_{ref}(y|x)$ for $y \sim \pi_\theta$.)
2. **Length bias.** Compute the Pearson correlation between RM score and response length on all 72 responses.
   Is the RM partly a length detector? Construct two pairs that would expose it.
3. **Flip the labels** on 25% of training pairs (simulated annotator noise) and retrain the RM. How fast does held-out
   accuracy degrade?
4. **From critiques to preferences.** Notebook 06 produces (original, revised) pairs via a constitution. Feed them to
   `DPOTrainer` as (rejected, chosen). That is a miniature version of RLAIF.

## Beyond the paper — a mini research question

*Is DPO's implicit reward a good reward model?* DPO's title claims "your language model is secretly a reward model".
Test it: use $\beta\log\frac{\pi_\theta}{\pi_{ref}}$ from the DPO policy as a classifier on held-out pairs, and compare its
accuracy and calibration with the explicit Bradley–Terry RM trained on the same data, as a function of training-set
size (8, 16, 28 pairs) and β. Where does each fail? (See also Lambert et al., *RewardBench*, 2024,
[arXiv:2403.13787](https://arxiv.org/abs/2403.13787), which evaluates DPO models as reward models.)

## References

* Christiano, P. et al. (2017). *Deep Reinforcement Learning from Human Preferences.* [arXiv:1706.03741](https://arxiv.org/abs/1706.03741)
* Stiennon, N. et al. (2020). *Learning to summarize from human feedback.* [arXiv:2009.01325](https://arxiv.org/abs/2009.01325)
* Ouyang, L. et al. (2022). *Training language models to follow instructions with human feedback.* [arXiv:2203.02155](https://arxiv.org/abs/2203.02155)
* Bai, Y. et al. (2022). *Training a Helpful and Harmless Assistant with RLHF.* [arXiv:2204.05862](https://arxiv.org/abs/2204.05862)
* Rafailov, R. et al. (2023). *Direct Preference Optimization.* [arXiv:2305.18290](https://arxiv.org/abs/2305.18290)
* Gao, L., Schulman, J., Hilton, J. (2022). *Scaling Laws for Reward Model Overoptimization.* [arXiv:2210.10760](https://arxiv.org/abs/2210.10760)
* Azar, M. G. et al. (2023). *A General Theoretical Paradigm to Understand Learning from Human Preferences* (IPO). [arXiv:2310.12036](https://arxiv.org/abs/2310.12036)
* Lambert, N. et al. (2024). *RewardBench.* [arXiv:2403.13787](https://arxiv.org/abs/2403.13787)
* Bradley, R. A., Terry, M. E. (1952). *Rank Analysis of Incomplete Block Designs.* Biometrika.